# ST-OMR Meter V3-A1 Sparse Colab

Bu notebook V3-A1 classification-only shadow deneyini **tam D10 local cache kopyası olmadan** çalıştırır. Yetkili D10 manifest/binding aynı kalır; tüm Meter etiketleri doğrulanır, fakat yerel SSD'ye yalnız tam 512 TRAIN replay görüntüsü + 1.224 Meter VALIDATION görüntüsü = 1.736 görüntü alınır. TEST/runtime/Resolver/production kapalıdır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json, os, shutil, subprocess, sys, time

REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_REF = 'fix/meter-real-domain-adaptation-v3-a1'
WORK_ROOT = Path('/content/st-omr-meter-v3-a1-sparse')
REPO_DIR = WORK_ROOT / 'repo'
TEACHER_BUNDLE = WORK_ROOT / 'teacher-gold-bundle-v1'
D10_SPARSE_CACHE = Path('/content/st-omr-meter-v3-a1-sparse-cache')
D11_LOCAL_CHECKPOINT = WORK_ROOT / 'd11-checkpoint.pt'
DRIVE_PILOT_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/00_AUDIT/teacher_gold_pilot_v1')
D10_DRIVE_ROOT = Path('/content/drive/MyDrive/ST-OMR-D10/stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a')
D11_DRIVE_CHECKPOINT = Path('/content/drive/MyDrive/ST-OMR-D11-authoritative/c8bc47bab1b8bccad77f42b8b5bdaa499ec0cc96/72132bd250865f350dd88229932b5b00dc9fa01041b0443992b7fa85613cacba/checkpoint-cd2d6192411371628518f4a8327cb0169910425494fa4a82082cd268d85254f3.pt')
D10_MANIFEST_SHA256 = '6927e1bcc5251257a983a306e2f1875c9515f97c6724a8fe9f24382c6ff30db4'
D10_ARTIFACT_BINDING_SHA256 = 'b72e2f5550c727484ea7226561fcd7c8e405d7d83a5bbab199d2780b8bc5db4d'
D11_CHECKPOINT_SHA256 = 'cd2d6192411371628518f4a8327cb0169910425494fa4a82082cd268d85254f3'
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS')
CONTROL_ROOT = DRIVE_RUNS_ROOT / 'meter-v3-a1-sparse-control'
STATUS_PATH = CONTROL_ROOT / 'status.json'
LOG_PATH = CONTROL_ROOT / 'runner.log'
RUN_ROOT = DRIVE_RUNS_ROOT / 'meter-real-domain-v3-a1-sparse-run'

def read_status():
    try:
        return json.loads(STATUS_PATH.read_text('ascii'))
    except (FileNotFoundError, OSError, UnicodeError, json.JSONDecodeError):
        return None

def heartbeat_age_seconds(status):
    if not status or not status.get('updated_at'):
        return float('inf')
    stamp = datetime.fromisoformat(status['updated_at'].replace('Z', '+00:00'))
    return (datetime.now(timezone.utc) - stamp).total_seconds()


## Prepare — tam 44k cache kopyası yasak


In [ ]:
DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
CONTROL_ROOT.mkdir(parents=True, exist_ok=True)
prior = read_status()
if prior and prior.get('state') == 'RUNNING' and heartbeat_age_seconds(prior) < 120:
    raise RuntimeError('Sparse V3-A1 run zaten aktif. Yalnız monitor hücresini çalıştırın.')
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', '--filter=blob:none', REPO_URL, str(REPO_DIR)], check=True)
repository_sha = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
required = [
    REPO_DIR / 'st_omr_training/meter_v3_a1_sparse_d10.py',
    REPO_DIR / 'tools/meter_real_domain_background_runner_v3_a1_sparse.py',
    REPO_DIR / 'st_omr_training/meter_real_domain_adaptation_v3_a1_run.py',
]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--extra-index-url', 'https://download.pytorch.org/whl/cpu', '-r', str(REPO_DIR / 'requirements-training.txt')], check=True)
pilot_inputs = {
    'pilot': DRIVE_PILOT_ROOT / 'pilot-data.json',
    'choices': DRIVE_PILOT_ROOT / 'ST_OMR_METER_TEACHER_GOLD_PILOT_choices.json',
    'permission': DRIVE_PILOT_ROOT / 'meter-training-permission-evidence-v1.json',
    'privacy': DRIVE_PILOT_ROOT / 'meter-privacy-review-evidence-v1.json',
}
for path in (*pilot_inputs.values(), D10_DRIVE_ROOT / 'COMPLETE', D10_DRIVE_ROOT / 'manifest.json', D10_DRIVE_ROOT / 'manifest.sha256', D10_DRIVE_ROOT / 'receipt.json', D11_DRIVE_CHECKPOINT):
    if not path.is_file():
        raise FileNotFoundError(path)
print(json.dumps({
    'experiment': 'meter-real-domain-adaptation-v3-a1-sparse',
    'branch': REPO_REF,
    'repository_sha': repository_sha,
    'full_cache_copy': False,
    'planned_local_images': 1736,
    'bbox_frozen_exact': True,
    'test_opened': False,
}, indent=2))


## Start / Resume Sparse V3-A1


In [ ]:
if (RUN_ROOT / 'RUN_COMPLETE').is_file():
    print({'already_complete': True, 'run_root': str(RUN_ROOT)})
    background_pid = None
else:
    prior = read_status()
    if prior and prior.get('state') == 'RUNNING' and heartbeat_age_seconds(prior) < 120:
        print({'already_running': True, 'status': str(STATUS_PATH)})
        background_pid = None
    else:
        command = [
            sys.executable, '-u', str(REPO_DIR / 'tools/meter_real_domain_background_runner_v3_a1_sparse.py'),
            '--repository-root', str(REPO_DIR),
            '--pilot', str(pilot_inputs['pilot']), '--choices', str(pilot_inputs['choices']),
            '--permission', str(pilot_inputs['permission']), '--privacy', str(pilot_inputs['privacy']),
            '--teacher-bundle', str(TEACHER_BUNDLE),
            '--d10-drive-root', str(D10_DRIVE_ROOT),
            '--d10-sparse-cache-root', str(D10_SPARSE_CACHE),
            '--d10-manifest-sha256', D10_MANIFEST_SHA256,
            '--d10-artifact-binding-sha256', D10_ARTIFACT_BINDING_SHA256,
            '--d11-drive-checkpoint', str(D11_DRIVE_CHECKPOINT),
            '--d11-local-checkpoint', str(D11_LOCAL_CHECKPOINT), '--d11-sha256', D11_CHECKPOINT_SHA256,
            '--output-root', str(RUN_ROOT), '--status-path', str(STATUS_PATH),
        ]
        log_handle = LOG_PATH.open('a', encoding='utf-8')
        log_handle.write(f'\n=== V3-A1 SPARSE launch {datetime.now(timezone.utc).isoformat()} repo={repository_sha} ===\n')
        log_handle.flush()
        environment = dict(os.environ)
        environment['PYTHONPATH'] = str(REPO_DIR)
        process = subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT, env=environment, start_new_session=True)
        background_pid = process.pid
        log_handle.close()
        print({'background_started': True, 'pid': background_pid, 'full_cache_copy': False, 'status': str(STATUS_PATH), 'log': str(LOG_PATH)})


## Live monitor


In [ ]:
from IPython.display import clear_output
while True:
    status = read_status()
    clear_output(wait=True)
    if not status:
        print('Sparse V3-A1 durum dosyası bekleniyor...')
        time.sleep(3)
        continue
    age = heartbeat_age_seconds(status)
    print(f"DURUM: {status.get('state')} | AŞAMA: {status.get('phase_index', 0)}/{status.get('phase_total', 9)} {status.get('phase')}")
    print(f"EPOCH: {status.get('epoch', status.get('completed_epoch', 0))}/{status.get('epochs_total', 20)} | BATCH: {status.get('batch', 0)}/{status.get('batches_total', 0)}")
    if status.get('files_total'):
        print(f"SPARSE İLERLEME: {status.get('files_completed', 0)}/{status['files_total']} | full_cache_copy={status.get('full_cache_copy', False)}")
    print(f"SÜRE: {status.get('elapsed_seconds', 0) // 60} dakika | HEARTBEAT: {int(age)} sn")
    print(f"OLAY: {status.get('event')}")
    if status.get('result'):
        print(f"SONUÇ: {status['result']}")
    if status.get('error'):
        print(f"HATA: {status.get('error_type')}: {status['error']}")
    if status.get('state') in {'COMPLETE', 'FAILED'}:
        break
    if age > 120:
        print('UYARI: heartbeat 120 saniyeyi geçti; runtime kopmuş olabilir.')
        break
    time.sleep(5)


## Bounded result


In [ ]:
def show_bounded_result():
    status = read_status()
    if not status or status.get('state') != 'COMPLETE':
        print(json.dumps(status or {'state': 'NO_STATUS'}, indent=2, sort_keys=True))
        if LOG_PATH.is_file():
            print('\nSON 30 LOG SATIRI:')
            print('\n'.join(LOG_PATH.read_text('utf-8', errors='replace').splitlines()[-30:]))
        return None
    metrics_files = sorted(RUN_ROOT.glob('metrics-*.json'))
    if len(metrics_files) != 1:
        raise RuntimeError(f'Final metrics dosyası sayısı beklenmedik: {len(metrics_files)}')
    metrics = json.loads(metrics_files[0].read_text('ascii'))
    if metrics.get('adaptation_version') != 'meter-real-domain-adaptation-v3-a1':
        raise RuntimeError(f'Yanlış adaptation sonucu: {metrics.get("adaptation_version")}')
    if metrics.get('bbox_frozen_exact') is not True:
        raise RuntimeError('V3-A1 bbox_frozen_exact true değil')
    summary = {
        'status': metrics['status'],
        'best_epoch': metrics['best']['epoch'],
        'baseline_real_macro_f1': metrics['baseline']['real_validation']['macro_f1'],
        'best_real_macro_f1': metrics['best']['real_validation']['macro_f1'],
        'best_real_accuracy': metrics['best']['real_validation']['accuracy'],
        'best_real_per_class_recall': metrics['best']['real_validation']['per_class_recall'],
        'baseline_synthetic_macro_f1': metrics['baseline']['synthetic_validation']['macro_f1'],
        'best_synthetic_macro_f1': metrics['best']['synthetic_validation']['macro_f1'],
        'baseline_synthetic_localization_f1': metrics['baseline']['synthetic_validation']['positive_localization_f1_2px'],
        'best_synthetic_localization_f1': metrics['best']['synthetic_validation']['positive_localization_f1_2px'],
        'bbox_frozen_exact': metrics['bbox_frozen_exact'],
        'full_cache_copy': False,
        'gate': metrics['best']['gate'],
        'checkpoint_sha256': metrics['best']['checkpoint_sha256'],
        'test_opened': metrics['test_opened'],
        'runtime_connected': metrics['runtime_connected'],
        'production_promotion_authorized': metrics['production_promotion_authorized'],
    }
    print(json.dumps(summary, indent=2, sort_keys=True))
    return summary

bounded_result = show_bounded_result()
